# Clase 9 · Riesgo de crédito: XGBoost, LightGBM y CatBoost, y el lift en cinco grupos

**Machine Learning** · Programa de Especialización Data Scientist · Sesión 9 de 14

Docente: **Josef Renato Rodriguez**

---

## Dónde quedamos

En la Clase 8, con el caso del banco, ningún boosting le ganó a la regresión logística: la señal estaba casi toda en una
pregunta, «¿ya lo contactamos antes?», y no había curvas ni interacciones que exprimir.

Hoy cambiamos de caso y de pregunta. Somos el equipo de riesgos de un banco con una cartera de **tarjetas de crédito**, y
cada mes queremos saber **qué clientes van a caer en default en los próximos tres meses**. Es el modelo de comportamiento
que todo banco tiene en producción, y responde tres preguntas:

1. **¿Cómo se arma la base?** Una foto por cliente y mes, un target que mira al futuro y un corte temporal 2024 · 2025 · 2026.
2. **¿Aquí sí ganan los boostings?** XGBoost, LightGBM y CatBoost contra la logística.
3. **¿El score ordena?** El lift en cinco grupos, de «Muy alto» a «Muy bajo»: cómo se cortan, cómo se leen y cómo debe
   ordenar la tasa de default de un grupo a otro.

## Qué vas a hacer hoy

| Bloque | Fase CRISP-DM | Contenido | Min |
|---|---|---|---|
| 1 | Entender los datos | La cartera, el target a 3 meses y el corte temporal | 20 |
| 2 | Modelar | XGBoost, LightGBM y CatBoost contra la logística | 35 |
| 3 | Evaluar | El lift en cinco grupos: cortes, tabla y orden | 35 |
| 4 | Monitorear | ¿Los grupos se mantienen en 2026? PSI y calibración por grupo | 20 |
| 5 | | Cierre: los números del día y preguntas tipo examen | 10 |

Igual que siempre: completa las celdas **`TU CÓDIGO`** y ejecuta la **`VERIFICACIÓN`** de abajo.
El notebook completo tarda menos de un minuto en Colab. La primera vez, la Celda 0 instala CatBoost.

---
## Celda 0 · Librerías y datos

La celda descarga `cartera_tarjetas.csv` del repositorio del curso (carpeta `Clase_9/Data`) y deja listas estas piezas:

- `cartera`: la base completa, **una fila por cliente y mes**.
- `NUM`, `CAT` y `X`: las 8 variables explicativas. `id_cliente` y `periodo` identifican la fila, pero **no entran al modelo**.
- `codificar(df)`: prepara las variables para XGBoost y LightGBM (one-hot de `tipo_empleo`; los vacíos se quedan como `NaN`).
- `metricas(y, p)`: AUC, Gini y KS, las tres métricas que pide un área de riesgos.
- `BANDAS`: los nombres de los cinco grupos, **de menor a mayor riesgo**.

El diccionario de variables está en `Clase_9/Data/diccionario_cartera_tarjetas.md`.

In [ ]:
import os, time, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
for paquete in ["xgboost", "lightgbm", "catboost"]:   # Colab trae xgboost y lightgbm; catboost se instala la primera vez
    try:
        __import__(paquete)
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings("ignore")

SEED = 42
NAVY, BLUE, MAG, GREEN, GRAY = "#0A2559", "#1A56E8", "#E6115E", "#12B886", "#5B6B87"
plt.rcParams.update({
    "figure.figsize": (7, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

def check(nombre, obtenido, esperado, tol=1e-3):
    if obtenido is None:
        print(f"[ ] {nombre}: todavía no calculaste nada")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}")
    print(f"     tu resultado: {float(obtenido):.4f}   |   esperado: {float(esperado):.4f}")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"     pista: {pista}")
    return bool(cond)

# ── Datos: la cartera de tarjetas, del repositorio del curso ──────────────────
ARCHIVO = "cartera_tarjetas.csv"
URL = "https://raw.githubusercontent.com/josefrodrim/Super_ML_course/main/Clase_9/Data/cartera_tarjetas.csv"
if not os.path.exists(ARCHIVO):
    try:
        urllib.request.urlretrieve(URL, ARCHIVO)
        print(f"{ARCHIVO} descargado del repositorio del curso")
    except Exception as e:
        raise FileNotFoundError(f"No se pudo descargar ({e}). Sube {ARCHIVO} al panel de archivos de Colab y vuelve a ejecutar.")
cartera = pd.read_csv(ARCHIVO)

ID, TIEMPO, TARGET = "id_cliente", "periodo", "default_3m"
NUM = ["edad", "ingreso_mensual", "antiguedad_meses", "utilizacion_tc",
       "dias_atraso_max_6m", "num_consultas_3m", "ratio_deuda_ingreso"]
CAT = ["tipo_empleo"]
X = NUM + CAT                                   # id_cliente y periodo NO entran al modelo
BANDAS = ["Muy bajo", "Bajo", "Medio", "Alto", "Muy alto"]   # de menor a mayor riesgo

def codificar(df):
    """Para XGBoost y LightGBM: numéricas tal cual (los NaN se quedan, ellos los manejan) y tipo_empleo en one-hot."""
    d = df[NUM].copy()
    for v in ["dependiente", "independiente", "informal"]:
        d[f"empleo_{v}"] = (df["tipo_empleo"] == v).astype(int)
    return d

def ks(y, p):
    fpr, tpr, _ = roc_curve(y, p)
    return (tpr - fpr).max()

def metricas(y, p):
    auc = roc_auc_score(y, p)
    return {"AUC": auc, "Gini": 2 * auc - 1, "KS": ks(y, p)}

print(f"{len(cartera):,} filas · {cartera[ID].nunique():,} clientes · períodos {cartera[TIEMPO].min()} a {cartera[TIEMPO].max()}".replace(",", " "))
print(f"Tasa de default a 3 meses en toda la cartera: {cartera[TARGET].mean():.2%}")
cartera.head()

---
# Bloque 1 · La cartera, el target a 3 meses y el corte temporal  ·  20 min

Cada fila es la **foto de un cliente al cierre de un mes**: cuánto usa de su línea, cuántos días de atraso tuvo en los últimos
seis meses, cuántas veces lo consultaron en la central de riesgos, etc. El target mira hacia adelante:

$$\text{default\_3m} = 1 \iff \text{el cliente llega a 90 o más días de atraso en alguno de los 3 meses siguientes a la foto}$$

Dos consecuencias que hay que tener presentes:

- **El target tarda en madurar.** La foto de junio de 2026 necesita julio, agosto y septiembre para saber si cayó.
  Por eso la base termina en **2026-06** aunque hoy estemos en octubre: las fotos de julio en adelante todavía no tienen target.
- **Un mismo cliente aparece en muchos meses.** Con un corte aleatorio, el modelo vería a un cliente en marzo y lo «predeciría» en abril.

El corte de hoy es por año:

| Muestra | Períodos | Para qué |
|---|---|---|
| **entrenamiento** | 2024-01 a 2024-12 | ajustar los modelos y **fijar los cortes de los cinco grupos** |
| **test** | 2025-01 a 2025-12 | parada temprana y elegir hiperparámetros |
| **OOT** | 2026-01 a 2026-06 | el juez final: se mira una vez |

### Ejercicio 1.1 — El corte temporal

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
anio = cartera[TIEMPO].str[:4].astype(int)     # el año de cada foto
ent = None     # las filas de 2024: cartera[anio == 2024]
test = None    # las de 2025
oot = None     # las de 2026

if oot is not None:
    for nombre, df in [("entrenamiento 2024", ent), ("test 2025", test), ("OOT 2026-S1", oot)]:
        print(f"{nombre:>18}: {len(df):>6} filas · {df[ID].nunique():>5} clientes · tasa de default {df[TARGET].mean():.2%}")

In [ ]:
# ── VERIFICACIÓN 1.1 ─────────────────────────────────────────────────────
r = [check("filas de entrenamiento", None if ent is None else len(ent), 21600, tol=0),
     check("filas del OOT", None if oot is None else len(oot), 11400, tol=0),
     check("tasa de default en entrenamiento", None if ent is None else ent[TARGET].mean(), 0.0609, tol=0.0005),
     check("tasa de default en el OOT", None if oot is None else oot[TARGET].mean(), 0.0968, tol=0.0005)]
print()
print("1.1 OK" if all(r) else "Revisa 1.1")

In [ ]:
# ── DEMOSTRACIÓN 1.2 ─────────────────────────────────────────────────────
mensual = cartera.groupby(TIEMPO)[TARGET].mean()
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(range(len(mensual)), mensual.values, "o-", color=NAVY, ms=4)
for desde, hasta, color, nombre in [(0, 11, GRAY, "entrenamiento"), (12, 23, BLUE, "test"), (24, len(mensual) - 1, MAG, "OOT")]:
    ax.axvspan(desde - 0.5, hasta + 0.5, color=color, alpha=0.10)
    ax.text((desde + hasta) / 2, mensual.max() * 1.08, nombre, ha="center", color=color, fontweight="bold")
ax.set_xticks(range(0, len(mensual), 3)); ax.set_xticklabels(mensual.index[::3], rotation=45)
ax.set_ylim(0, mensual.max() * 1.18); ax.set_ylabel("tasa de default a 3 meses")
ax.set_title("La tasa es estable en 2024 y 2025, y sube en 2026")
plt.tight_layout(); plt.show()

La tasa ronda el **6 %** en 2024 y 2025, y en 2026 sube a **9.7 %**. El OOT no es «más de lo mismo»: es un año más difícil.
Vamos a ver en el Bloque 4 qué le pasa al score cuando el entorno cambia.

### Ejercicio 1.3 — ¿Cuántos clientes del test ya estaban en el entrenamiento?

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
repetidos = None           # proporción de filas del test cuyo id_cliente está en ent: test[ID].isin(ent[ID]).mean()
fotos_por_cliente = None   # cuántas filas tiene cada cliente, en promedio: cartera.groupby(ID).size().mean()

In [ ]:
# ── VERIFICACIÓN 1.3 ─────────────────────────────────────────────────────
r = [check("filas del test con cliente ya visto", repetidos, 0.9268, tol=0.002),
     check("fotos por cliente", fotos_por_cliente, 6.076, tol=0.05)]
print()
print("Bloque 1 COMPLETO" if all(r) else "Revisa 1.3")

El **93 %** de las filas del test son de clientes conocidos. Eso no es un problema en un modelo de comportamiento: en
producción también se puntúa a clientes que el banco ya conoce. El problema sería **mezclar meses**: con `train_test_split`
aleatorio, la foto de marzo de un cliente quedaría en entrenamiento y la de abril en test, y el test se vería mejor de lo que
va a funcionar el modelo en el futuro.

> **Regla:** `id_cliente` y `periodo` organizan la base, pero no son variables. Si el `id_cliente` entrara al modelo, el árbol
> aprendería «el cliente 104 377 cae» y no serviría para ningún cliente nuevo.

### Preparación — Las matrices de cada muestra

XGBoost y LightGBM reciben `codificar(df)`; CatBoost recibe las variables originales, con `tipo_empleo` como texto.

In [ ]:
# ── PREPARACIÓN (ejecútala tal cual) ──────────────────────────────────────
if oot is None:   # si el 1.1 quedó sin completar, se usa el corte de referencia para poder seguir
    print("El 1.1 está incompleto: uso el corte 2024 · 2025 · 2026 para seguir con la clase.")
    ent, test, oot = cartera[anio == 2024], cartera[anio == 2025], cartera[anio == 2026]
if oot is not None:
    y_ent, y_test, y_oot = ent[TARGET].values, test[TARGET].values, oot[TARGET].values
    Xe, Xt, Xo = codificar(ent), codificar(test), codificar(oot)      # para XGBoost y LightGBM
    Se, St, So = ent[X], test[X], oot[X]                              # para CatBoost (y la logística)
    print(f"XGBoost y LightGBM ven {Xe.shape[1]} columnas: {list(Xe.columns)}")

---
# Bloque 2 · XGBoost, LightGBM y CatBoost contra la logística  ·  35 min

Las reglas son las de la Clase 8:

1. Se entrena en 2024.
2. La **parada temprana** y los hiperparámetros se eligen con el **test de 2025**. Desde ese momento el test ya no es imparcial.
3. El **OOT de 2026 se mira una vez**, al final, para todos los modelos por igual.

En riesgos se reporta el **Gini** ($2 \cdot \text{AUC} - 1$) y el **KS**: la mayor distancia entre la distribución del score
de los buenos y la de los malos.

### Demostración 2.0 — La marca a batir: la regresión logística

In [ ]:
# ── DEMOSTRACIÓN 2.0 ─────────────────────────────────────────────────────
if oot is not None:
    logit = make_pipeline(
        ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
                           ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)]),
        LogisticRegression(max_iter=2000)).fit(Se, y_ent)
    p_log = {"ent": logit.predict_proba(Se)[:, 1], "test": logit.predict_proba(St)[:, 1], "oot": logit.predict_proba(So)[:, 1]}
    print("Regresión logística · test:", {k: round(v, 3) for k, v in metricas(y_test, p_log["test"]).items()})

### Ejercicio 2.1 — XGBoost con cuatro profundidades

Con `early_stopping_rounds=100`, XGBoost para cuando el AUC del test lleva 100 árboles sin mejorar y se queda con el mejor
punto. Prueba `max_depth` 2, 3, 4 y 6, y quédate con el de mayor AUC en el test.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
xgb_modelos, filas = {}, []
for prof in [2, 3, 4, 6]:
    m = XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=prof, subsample=0.8, colsample_bytree=0.8,
                      eval_metric="auc", early_stopping_rounds=100, random_state=SEED, n_jobs=1)
    m.fit(Xe, y_ent, eval_set=[(Xt, y_test)], verbose=False)
    xgb_modelos[prof] = m
    filas.append({"max_depth": prof, "árboles": m.best_iteration + 1,
                  "AUC ent": roc_auc_score(y_ent, m.predict_proba(Xe)[:, 1]),
                  "AUC test": None})     # el AUC de m en el test: roc_auc_score(y_test, m.predict_proba(Xt)[:, 1])
tabla_xgb = pd.DataFrame(filas)
mejor_prof = None   # la max_depth con mayor "AUC test"
xgb = None          # xgb_modelos[mejor_prof]

In [ ]:
# ── VERIFICACIÓN 2.1 ─────────────────────────────────────────────────────
r = [check("AUC test del XGBoost elegido", None if xgb is None else roc_auc_score(y_test, xgb.predict_proba(Xt)[:, 1]), 0.8412, tol=0.004),
     check("AUC test con max_depth = 6", tabla_xgb["AUC test"].iloc[3] if tabla_xgb["AUC test"].notna().all() else None, 0.8367, tol=0.004),
     check_bool("el elegido es poco profundo (2 o 3 niveles)", mejor_prof in (2, 3), "¿elegiste con idxmax del AUC test?")]
print()
print("2.1 OK" if all(r) else "Revisa 2.1")

Árboles de **2 niveles** y 140 árboles. Con 6 niveles el entrenamiento sube y el test baja: sobreajuste.
La diferencia entre 2 y 3 niveles es de milésimas, así que en tu Colab puede salir cualquiera de los dos.

### Ejercicio 2.2 — LightGBM con tres tamaños de hoja

LightGBM hace crecer cada árbol **por hojas** hasta `num_leaves`. `min_child_samples=50` pide al menos 50 clientes por hoja.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
lgb_modelos, filas = {}, []
for hojas in [8, 15, 31]:
    m = LGBMClassifier(n_estimators=2000, learning_rate=0.05, num_leaves=hojas, min_child_samples=50, subsample=0.8,
                       subsample_freq=1, colsample_bytree=0.8, random_state=SEED, n_jobs=1, verbose=-1)
    m.fit(Xe, y_ent, eval_set=[(Xt, y_test)], eval_metric="auc", callbacks=[lgb.early_stopping(100, verbose=False)])
    lgb_modelos[hojas] = m
    filas.append({"num_leaves": hojas, "árboles": m.best_iteration_,
                  "AUC ent": roc_auc_score(y_ent, m.predict_proba(Xe)[:, 1]),
                  "AUC test": None})     # el AUC de m en el test
tabla_lgb = pd.DataFrame(filas)
mejor_hojas = None  # el num_leaves con mayor "AUC test"
lgbm = None         # lgb_modelos[mejor_hojas]

In [ ]:
# ── VERIFICACIÓN 2.2 ─────────────────────────────────────────────────────
r = [check("AUC test del LightGBM elegido", None if lgbm is None else roc_auc_score(y_test, lgbm.predict_proba(Xt)[:, 1]), 0.8404, tol=0.004),
     check("AUC entrenamiento con 31 hojas", tabla_lgb["AUC ent"].iloc[2], 0.8612, tol=0.01)]
print()
print("2.2 OK" if all(r) else "Revisa 2.2")

### Ejercicio 2.3 — CatBoost con la categórica en texto

CatBoost recibe `tipo_empleo` como texto en `cat_features` y lo convierte en una tasa con las **estadísticas ordenadas** de la
Clase 8. Los `NaN` del ingreso también los maneja solo.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
cb_modelos, filas = {}, []
for prof in [4, 6]:
    m = CatBoostClassifier(iterations=2000, learning_rate=0.05, depth=prof, eval_metric="AUC", early_stopping_rounds=100,
                           cat_features=CAT, random_seed=SEED, thread_count=1, verbose=0)
    m.fit(Se, y_ent, eval_set=(St, y_test), use_best_model=True)
    cb_modelos[prof] = m
    filas.append({"depth": prof, "árboles": m.get_best_iteration() + 1,
                  "AUC ent": roc_auc_score(y_ent, m.predict_proba(Se)[:, 1]),
                  "AUC test": None})     # el AUC de m en el test (ojo: CatBoost usa St, no Xt)
tabla_cb = pd.DataFrame(filas)
mejor_depth = None  # la depth con mayor "AUC test"
catb = None         # cb_modelos[mejor_depth]

In [ ]:
# ── VERIFICACIÓN 2.3 ─────────────────────────────────────────────────────
r = [check("AUC test del CatBoost elegido", None if catb is None else roc_auc_score(y_test, catb.predict_proba(St)[:, 1]), 0.8411, tol=0.004)]
print()
print("2.3 OK" if all(r) else "Revisa 2.3")

### Ejercicio 2.4 — La tabla de los cuatro modelos

Guarda las probabilidades de cada modelo en las tres muestras en el diccionario `p` y arma la tabla con AUC, Gini y KS en el
test y en el OOT.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
p = {"Regresión logística": p_log,
     "XGBoost": None if xgb is None else {"ent": xgb.predict_proba(Xe)[:, 1], "test": xgb.predict_proba(Xt)[:, 1], "oot": xgb.predict_proba(Xo)[:, 1]},
     "LightGBM": None,   # lo mismo con lgbm (usa Xe, Xt, Xo)
     "CatBoost": None}   # lo mismo con catb (usa Se, St, So)

tabla_modelos = None     # una fila por modelo con metricas(y_test, s["test"]) y metricas(y_oot, s["oot"])

In [ ]:
# ── VERIFICACIÓN 2.4 ─────────────────────────────────────────────────────
def celda_m(modelo, col):
    return None if tabla_modelos is None else tabla_modelos.loc[modelo, col]
r = [check("Gini OOT · regresión logística", celda_m("Regresión logística", "Gini OOT"), 0.6368, tol=0.006),
     check("Gini OOT · XGBoost", celda_m("XGBoost", "Gini OOT"), 0.6766, tol=0.008),
     check("Gini OOT · LightGBM", celda_m("LightGBM", "Gini OOT"), 0.6727, tol=0.008),
     check("Gini OOT · CatBoost", celda_m("CatBoost", "Gini OOT"), 0.6729, tol=0.008),
     check("KS OOT · XGBoost", celda_m("XGBoost", "KS OOT"), 0.5446, tol=0.01)]
print()
print("Bloque 2 COMPLETO" if all(r) else "Revisa 2.4")

| Modelo | Gini test | Gini OOT | KS OOT |
|---|---|---|---|
| Regresión logística | 0.644 | 0.637 | 0.500 |
| XGBoost | 0.682 | **0.677** | 0.545 |
| LightGBM | 0.681 | 0.673 | 0.540 |
| CatBoost | 0.682 | 0.673 | 0.537 |

- **Aquí los boostings sí le ganan a la logística:** unos **4 puntos de Gini** en 2026. En la Clase 8 no pasó. ¿Qué cambió?
  En esta cartera el riesgo tiene **curvas e interacciones**: el riesgo se dispara cuando la utilización pasa de ~60 %, y la deuda
  alta pesa más en los informales. Una logística con las variables tal cual no ve esas curvas; los árboles, sí.
- **Los tres boostings empatan** (Gini entre 0.673 y 0.677). Las diferencias son de milésimas y cambian con la semilla.
- **El Gini baja poco de 2025 a 2026**, aunque la tasa de default subió de 6.3 % a 9.7 %: el score sigue **ordenando** igual de bien.
  Si las probabilidades siguen siendo correctas es otra pregunta (Bloque 4).

**El modelo campeón:** con tres boostings empatados, elegimos **XGBoost**: es el más documentado, el más fácil de poner en
producción y el que vamos a explicar con SHAP en la Clase 10. LightGBM y CatBoost quedan como retadores.

---
# Bloque 3 · El lift en cinco grupos  ·  35 min

Un comité de créditos no trabaja con «p = 0.137». Trabaja con **grupos de riesgo**: «a los *Muy alto* no se les sube la línea;
a los *Muy bajo* se les ofrece un incremento». Los cinco grupos salen de cortar el score en **quintiles**.

### Cómo se cortan

1. Se ordena a los clientes del **entrenamiento** por su probabilidad de default.
2. Los cuatro cortes son los percentiles **20, 40, 60 y 80**: cada grupo tiene el 20 % de los clientes de 2024.
3. Los cortes **se guardan con el modelo y no se mueven**: un cliente de 2026 con p = 0.05 cae en el grupo donde habría caído un cliente de 2024 con p = 0.05.

### Cómo debe ordenar

| Grupo | Probabilidad del modelo | Tasa de default observada | Lift |
|---|---|---|---|
| **Muy alto** | el 20 % con mayor p | la más alta | mucho mayor que 1 |
| Alto | | ↓ | |
| Medio | | ↓ | cerca de 1 o menos |
| Bajo | | ↓ | |
| **Muy bajo** | el 20 % con menor p | la más baja | cerca de 0 |

Un score sirve si cumple tres cosas: **la tasa baja de un grupo al siguiente** (orden monótono), **el grupo de arriba concentra
la mayoría de los defaults** (captura) y **la probabilidad media de cada grupo se parece a su tasa real** (calibración).

### Ejercicio 3.1 — Los cortes, en el entrenamiento

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
p_campeon = p["XGBoost"]
cortes = None     # los percentiles 20, 40, 60 y 80 de p_campeon["ent"]: np.quantile(..., [0.2, 0.4, 0.6, 0.8])
bordes = None     # np.r_[-np.inf, cortes, np.inf]

def asignar_grupo(score):
    """El grupo de riesgo de cada probabilidad, con los cortes fijos del entrenamiento."""
    return None   # pd.cut(score, bordes, labels=BANDAS, right=False)

if bordes is not None:
    for nombre, desde, hasta in zip(BANDAS, bordes[:-1], bordes[1:]):
        print(f"{nombre:>9}: probabilidad desde {desde:.4f} hasta {hasta:.4f}")

In [ ]:
# ── VERIFICACIÓN 3.1 ─────────────────────────────────────────────────────
r = [check("corte entre Muy bajo y Bajo (percentil 20)", None if cortes is None else cortes[0], 0.0142, tol=0.002),
     check("corte entre Alto y Muy alto (percentil 80)", None if cortes is None else cortes[3], 0.0897, tol=0.006),
     check_bool("los bordes empiezan en -inf y terminan en +inf", bordes is not None and np.isinf(bordes[0]) and np.isinf(bordes[-1]))]
print()
print("3.1 OK" if all(r) else "Revisa 3.1")

Fíjate en lo asimétrico: los cuatro grupos de abajo caben entre p = 0 y p = 0.090, y *Muy alto* va de 0.090 hasta casi 1.
Es lo normal en riesgo de crédito: la mayoría de los clientes es buena y **el riesgo se concentra en la cola**.

### Ejercicio 3.2 — La tabla de lift

Escribe `tabla_lift(y, score)`. Las filas van de *Muy alto* a *Muy bajo*.

| Columna | Cálculo |
|---|---|
| clientes, defaults | conteos por grupo |
| p media | promedio de la probabilidad del modelo en el grupo |
| tasa | defaults / clientes |
| lift | tasa del grupo / tasa de toda la muestra |
| captura acum. | defaults acumulados desde *Muy alto* / defaults totales |

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
def tabla_lift(y, score):
    d = pd.DataFrame({"grupo": asignar_grupo(score), "y": y, "p": score})
    t = d.groupby("grupo", observed=False).agg(clientes=("y", "size"), defaults=("y", "sum"), p_media=("p", "mean")).iloc[::-1]
    t["% clientes"] = t["clientes"] / t["clientes"].sum()
    t["tasa"] = None            # defaults / clientes
    t["lift"] = None            # tasa / (defaults totales / clientes totales)
    t["captura acum."] = None   # t["defaults"].cumsum() / t["defaults"].sum()
    return t

lift_ent = None    # tabla_lift(y_ent, p_campeon["ent"])
lift_test = None   # lo mismo con el test
lift_oot = None    # lo mismo con el OOT

In [ ]:
# ── VERIFICACIÓN 3.2 ─────────────────────────────────────────────────────
def celda(t, g, col):
    return None if t is None or t[col].isna().any() else t.loc[g, col]
r = [check("tasa de Muy alto en el test", celda(lift_test, "Muy alto", "tasa"), 0.2185, tol=0.008),
     check("lift de Muy alto en el test", celda(lift_test, "Muy alto", "lift"), 3.4804, tol=0.12),
     check("captura de Muy alto en el test", celda(lift_test, "Muy alto", "captura acum."), 0.7006, tol=0.015),
     check("tasa de Muy bajo en el test", celda(lift_test, "Muy bajo", "tasa"), 0.0078, tol=0.003),
     check("tasa de Muy alto en el OOT", celda(lift_oot, "Muy alto", "tasa"), 0.272, tol=0.01)]
print()
print("3.2 OK" if all(r) else "Revisa 3.2")

In [ ]:
# ── DEMOSTRACIÓN 3.3 ─────────────────────────────────────────────────────
if lift_oot is not None and not lift_oot["tasa"].isna().any():
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
    x = np.arange(5)
    for k, (t, color, nombre) in enumerate([(lift_ent, GRAY, "entrenamiento 2024"), (lift_test, BLUE, "test 2025"), (lift_oot, MAG, "OOT 2026")]):
        barras = axes[0].bar(x + (k - 1) * 0.27, t["tasa"], width=0.27, color=color, label=nombre)
        axes[0].bar_label(barras, labels=[f"{v:.1%}" for v in t["tasa"]], fontsize=8, padding=2)
    axes[0].set_xticks(x); axes[0].set_xticklabels(lift_test.index.astype(str))
    axes[0].set_ylabel("tasa de default a 3 meses"); axes[0].set_title("La tasa baja de grupo en grupo"); axes[0].legend()

    for t, color, nombre in [(lift_test, BLUE, "test 2025"), (lift_oot, MAG, "OOT 2026")]:
        axes[1].plot([0] + list(100 * t["% clientes"].cumsum()), [0] + list(100 * t["captura acum."]), "o-", color=color, label=nombre)
    axes[1].plot([0, 100], [0, 100], "--", color=GRAY, label="al azar")
    axes[1].set_xlabel("% de clientes revisados, desde Muy alto"); axes[1].set_ylabel("% de los defaults capturados")
    axes[1].set_title("Curva de captura (ganancias)"); axes[1].legend()
    plt.tight_layout(); plt.show()

Cómo se lee la tabla de 2025:

- **Muy alto concentra el 70 % de los defaults** con el 20 % de los clientes: su tasa es **21.8 %**, 3.48 veces la cartera.
- **Muy bajo cae al 0.78 %:** de cada mil clientes, unos 8 entran en default. Ahí está el espacio para crecer.
- **Del primero al segundo grupo, la tasa cae a la cuarta parte** (21.8 % → 5.2 %). Es la cola del riesgo que vimos en el Ejercicio 3.1.

### Ejercicio 3.4 — ¿Ordena? La prueba de monotonía

Escribe `ordena(t)`: devuelve `True` si la tasa **baja estrictamente** de *Muy alto* a *Muy bajo*, y la lista de los pares
de grupos donde el orden se invierte.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
def ordena(t):
    tasas = t["tasa"].values                       # de Muy alto a Muy bajo
    inversiones = None   # los pares (t.index[i], t.index[i + 1]) donde tasas[i] <= tasas[i + 1]
    return None          # (len(inversiones) == 0, inversiones)

In [ ]:
# ── VERIFICACIÓN 3.4 ─────────────────────────────────────────────────────
def prueba(t):
    try:
        resultado = ordena(t)
        return resultado is not None and resultado[0]
    except Exception:
        return False
r = [check_bool("ordena en el entrenamiento", prueba(lift_ent), "¿completaste tabla_lift y ordena?"),
     check_bool("ordena en el test", prueba(lift_test))]
print()
print("3.4 OK" if all(r) else "Revisa 3.4")

In [ ]:
# ── DEMOSTRACIÓN 3.5 ─────────────────────────────────────────────────────
def wilson(k, n, z=1.645):
    """Intervalo del 90 % para una tasa k/n."""
    p_ = k / n
    centro = (p_ + z**2 / (2 * n)) / (1 + z**2 / n)
    radio = z * np.sqrt(p_ * (1 - p_) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centro - radio, centro + radio

if lift_oot is not None and not lift_oot["tasa"].isna().any():
    print("OOT 2026 · tasa por grupo con su intervalo del 90 %\n")
    for g in lift_oot.index:
        lo, hi = wilson(lift_oot.loc[g, "defaults"], lift_oot.loc[g, "clientes"])
        print(f"{g:>9}: {lift_oot.loc[g, 'tasa']:6.2%}  [{lo:6.2%}, {hi:6.2%}]  · {int(lift_oot.loc[g, 'defaults'])} defaults de {int(lift_oot.loc[g, 'clientes'])}")

En nuestra corrida, en 2026 **Bajo y Muy bajo quedan prácticamente empatados** (1.52 % y 1.53 %): una inversión de
centésimas, con unos 30 defaults en cada grupo. Los intervalos se solapan casi por completo, así que no es una señal de que el
modelo se rompió: es que **abajo hay muy pocos defaults para distinguir dos grupos**.

> **Cómo se reporta en un banco:** la prueba estricta («ordena / no ordena») se acompaña de los intervalos. Una inversión en
> grupos con pocos eventos se anota y se vigila. Una inversión arriba, entre *Muy alto* y *Alto*, sería grave: ahí están los defaults.
> Si los dos grupos de abajo empatan mes tras mes, se pueden **fusionar** en uno.

### Ejercicio 3.6 — ¿Qué modelo concentra más riesgo arriba?

Repite el corte por quintiles para cada modelo, **con sus propios cortes del entrenamiento**, y compara la captura de
*Muy alto* y la tasa de *Muy bajo* en el OOT.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
filas = []
for nombre, s in p.items():
    if s is None:
        continue
    b = np.r_[-np.inf, np.quantile(s["ent"], [0.2, 0.4, 0.6, 0.8]), np.inf]   # los cortes de ESTE modelo
    grupo = pd.cut(s["oot"], b, labels=BANDAS, right=False)
    arriba, abajo = np.asarray(grupo == "Muy alto"), np.asarray(grupo == "Muy bajo")
    filas.append({"modelo": nombre,
                  "captura Muy alto": None,   # y_oot[arriba].sum() / y_oot.sum()
                  "lift Muy alto": None,      # y_oot[arriba].mean() / y_oot.mean()
                  "tasa Muy bajo": None})     # y_oot[abajo].mean()
captura_modelos = pd.DataFrame(filas).set_index("modelo")

In [ ]:
# ── VERIFICACIÓN 3.6 ─────────────────────────────────────────────────────
def cap(m):
    if m not in captura_modelos.index:
        return None
    v = captura_modelos.loc[m, "captura Muy alto"]
    return None if v is None or pd.isna(v) else v
r = [check("captura de Muy alto · logística", cap("Regresión logística"), 0.7117, tol=0.015),
     check("captura de Muy alto · XGBoost", cap("XGBoost"), 0.7452, tol=0.015)]
print()
print("Bloque 3 COMPLETO" if all(r) else "Revisa 3.6")

Con el mismo 20 % de clientes revisados, **XGBoost captura el 74.5 % de los defaults de 2026 y la logística el 71.2 %**.
LightGBM y CatBoost quedan en el mismo rango, entre 74 % y 75 %. Esos 3.4 puntos son la versión de negocio de la diferencia de Gini del Bloque 2: de cada 100 clientes que van a caer en
default, el boosting pone 3 más en el grupo que el banco vigila.

---
# Bloque 4 · ¿Los grupos se mantienen en 2026?  ·  20 min

Dos preguntas distintas que hay que hacer cada mes:

1. **¿La población cambió?** El **PSI** compara qué parte de los clientes cae en cada grupo contra el 20 % del entrenamiento:

$$\text{PSI} = \sum_{i=1}^{5} (a_i - e_i)\,\ln\frac{a_i}{e_i}$$

donde $e_i$ es la proporción esperada (0.20) y $a_i$ la observada. Menos de 0.10 es estable; de 0.10 a 0.25, vigilar; más de 0.25, revisar el modelo.

2. **¿Las probabilidades siguen siendo correctas?** Se compara, grupo por grupo, la **probabilidad media del modelo** con la **tasa observada**.

### Ejercicio 4.1 — El PSI del test y del OOT

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
def psi(esperado, observado, eps=1e-4):
    e = np.clip(np.asarray(esperado, dtype=float), eps, None)
    a = np.clip(np.asarray(observado, dtype=float), eps, None)
    return None   # np.sum((a - e) * np.log(a / e)), como float

psi_test = None   # psi(lift_ent["% clientes"], lift_test["% clientes"])
psi_oot = None    # lo mismo con lift_oot

In [ ]:
# ── VERIFICACIÓN 4.1 ─────────────────────────────────────────────────────
r = [check("PSI entrenamiento → test", psi_test, 0.0024, tol=0.004),
     check("PSI entrenamiento → OOT", psi_oot, 0.0307, tol=0.01)]
print()
print("4.1 OK" if all(r) else "Revisa 4.1")

**PSI de 0.031 en 2026: estable.** En 2026 hay algo más de clientes en *Muy alto* (26.5 % en lugar de 20 %), pero lejos del umbral de 0.10.

### Ejercicio 4.2 — Probabilidad media contra tasa observada

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
calibracion = None   # un DataFrame con p_media y tasa de lift_test y de lift_oot, y la columna tasa OOT / p media OOT

In [ ]:
# ── VERIFICACIÓN 4.2 ─────────────────────────────────────────────────────
r = [check("p media de Muy alto en el OOT", None if calibracion is None else calibracion.loc["Muy alto", "p media OOT"], 0.2096, tol=0.008),
     check("tasa / p de Muy alto en el OOT", None if calibracion is None else calibracion.loc["Muy alto", "tasa / p (OOT)"], 1.2975, tol=0.08)]
print()
print("Bloque 4 COMPLETO" if all(r) else "Revisa 4.2")

Aquí está lo importante del día:

- **En el test, la probabilidad media y la tasa se parecen** grupo por grupo (20.0 % contra 21.8 % en *Muy alto*). El modelo está calibrado.
- **En 2026, no:** *Muy alto* tiene p media de **21.0 %** y una tasa real de **27.2 %**. En toda la cartera, el modelo
  esperaba 7.6 % de default y hubo 9.7 %.
- **Y el PSI no lo vio.** El PSI mide si cambió **quién** está en cada grupo, no **cuánto cae** cada grupo. En 2026 entraron
  más independientes e informales y subió la utilización, pero sobre todo **el año fue peor**, y más en los grupos de mayor riesgo.

| Lo que miras | 2026 | Qué dice |
|---|---|---|
| Gini | 0.677 (test 0.682) | el score **ordena** igual |
| PSI de los grupos | 0.031 | la población **casi no cambió** |
| Tasa / p media | 1.30 en *Muy alto* | las probabilidades **se quedaron cortas** |

> **El diagnóstico:** no hay que reentrenar (el orden se mantiene), hay que **recalibrar**: ajustar el nivel de las probabilidades
> sin tocar el orden. Para provisiones o precios, una PD que se queda corta un 30 % es un problema serio aunque el Gini esté perfecto.

---
# Bloque 5 · Cierre  ·  10 min

### Los números del día

| | Test 2025 | OOT 2026 |
|---|---|---|
| Tasa de default | 6.3 % | 9.7 % |
| Gini · logística | 0.644 | 0.637 |
| Gini · XGBoost (campeón) | 0.682 | 0.677 |
| Tasa de *Muy alto* · lift | 21.8 % · 3.48 | 27.2 % · 2.81 |
| Captura de *Muy alto* | 70 % | 75 % |
| Tasa de *Muy bajo* | 0.78 % | 1.53 % |
| ¿Ordena? | sí | casi: Bajo y Muy bajo empatan |
| PSI de los grupos | 0.002 | 0.031 |

### Checklist de salida

- [ ] Armo una base de comportamiento: una foto por cliente y mes, y un target que mira 3 meses hacia adelante.
- [ ] Sé por qué la base termina meses antes de hoy: el target tiene que madurar.
- [ ] Corto por tiempo, y no uso `id_cliente` ni `periodo` como variables.
- [ ] Entreno XGBoost, LightGBM y CatBoost con parada temprana, y comparo con Gini y KS en un OOT intacto.
- [ ] Sé explicar por qué aquí los boostings le ganan a la logística y en el banco de la Clase 8 no.
- [ ] Corto el score en cinco grupos con los quintiles del entrenamiento, y esos cortes no se mueven.
- [ ] Leo una tabla de lift: tasa, lift, captura acumulada y probabilidad media por grupo.
- [ ] Pruebo que la tasa baje de *Muy alto* a *Muy bajo*, con intervalos para los grupos con pocos defaults.
- [ ] Distingo tres cosas: el score ordena (Gini), la población cambió (PSI) y las probabilidades acertaron (calibración).

### Preguntas tipo examen

**1.** Una base tiene fotos hasta 2026-09 y hoy es octubre de 2026. El target es default en los 3 meses siguientes. ¿Qué fotos sirven para entrenar?

> **Respuesta.** Solo hasta **2026-06**: la foto de junio necesita julio, agosto y septiembre. Las de julio a septiembre no tienen el target completo; si se usan, se cuentan como «buenos» clientes que todavía podrían caer.

**2.** El grupo *Muy alto* tiene 4 348 clientes y 950 defaults; la cartera tiene 21 600 clientes y 1 356 defaults. Calcula la tasa, el lift y la captura.

> **Respuesta.** Tasa = 950 / 4 348 = **21.8 %**. Tasa de la cartera = 1 356 / 21 600 = 6.28 %. Lift = 21.8 / 6.28 = **3.48**. Captura = 950 / 1 356 = **70 %**.

**3.** ¿Por qué los cortes de los grupos se calculan en el entrenamiento y no de nuevo en cada mes?

> **Respuesta.** Si se recalculan cada mes, cada grupo tiene siempre el 20 % por construcción: el PSI da 0 y no se ve ningún cambio. Además el mismo score podría ser *Alto* un mes y *Muy alto* el siguiente.

**4.** En 2026 la tasa de Bajo es 1.52 % y la de Muy bajo 1.53 %. ¿El modelo dejó de funcionar?

> **Respuesta.** No. Son unos 30 defaults por grupo y los intervalos se solapan: la diferencia es ruido. El orden de arriba, donde están los defaults, se mantiene. Se anota, se vigila y, si se repite, se fusionan los dos grupos.

**5.** El Gini del modelo no cambió, el PSI es 0.03 y la tasa real de *Muy alto* es 30 % mayor que la probabilidad media. ¿Reentrenar, recalibrar o nada?

> **Respuesta.** **Recalibrar.** El orden está bien (Gini) y la población es la misma (PSI); lo que falló es el nivel de las probabilidades, porque el entorno empeoró.

**6.** ¿Por qué en este caso XGBoost le gana a la logística y en el banco de la Clase 8 no?

> **Respuesta.** Porque aquí el riesgo tiene curvas e interacciones (se dispara con la utilización alta; la deuda pesa más en los informales) que una logística con las variables tal cual no captura. En el banco, la señal estaba casi toda en una variable binaria.

### Lo que sigue

**Clase 10 · Explicabilidad e interpretabilidad.** Hoy vimos *qué tan bien* ordena XGBoost. La próxima clase preguntamos *por qué*:
importancia por permutación, PDP e ICE (¿dónde se dispara el riesgo con la utilización?) y SHAP para explicar por qué un
cliente quedó en *Muy alto*, como lo pediría un validador o un regulador.